In [1]:
import numpy as np
import pandas as pd
import polars as pl

pd.set_option('display.max_columns', None)

In [2]:
augmented_df = pl.read_parquet("./imputed_augmented_us-counties-states_latest_variants.parquet", low_memory=False)

In [3]:
augmented_df_pd = augmented_df.to_pandas().sort_values(["fips","days_from_start"])
augmented_df_pd["date"] = pd.to_datetime(augmented_df_pd["date"])
augmented_df_pd["datetime"] = pd.to_datetime(augmented_df_pd["datetime"])
augmented_df_pd = augmented_df_pd[augmented_df_pd["rolled_cases"] >= 20]

In [4]:
augmented_df_pd.columns

Index(['date', 'county', 'state', 'cases', 'deaths', 'datetime',
       'days_from_start', 'rolled_cases', 'LAT', 'LON',
       ...
       'Variant % P.2', 'Variant % R.1', 'Variant % XBB', 'Variant % XBB.1.16',
       'Variant % XBB.1.5', 'Variant % XBB.1.5.1', 'Variant % XBB.1.9.1',
       'Variant % XBB.1.9.2', 'Variant % XBB.2.3', 'fips'],
      dtype='object', length=478)

In [5]:
latest_day = augmented_df_pd["days_from_start"].max()
latest_day_parity = "odd" if ((latest_day % 2) == 1) else "even"
print(f"The latest day={latest_day} is {latest_day_parity}")

The latest day=1157.0 is odd


In [6]:
odd_data = augmented_df_pd[(augmented_df_pd["days_from_start"] % 2)==1]
even_data = augmented_df_pd[(augmented_df_pd["days_from_start"] % 2)==0]

### Generate log_shifted_rolled_cases

In [7]:
# cutoff is odd, it will use data from the even days 1 day before
# Group by FIPS and compute the difference
augmented_df_pd['log_rolled_cases'] = np.log(augmented_df_pd['rolled_cases'])
augmented_df_pd['log_shifted_rolled_cases'] = augmented_df_pd.groupby('fips')['log_rolled_cases'].diff()
augmented_df_pd = augmented_df_pd.dropna(subset=["log_shifted_rolled_cases"])

In [9]:
augmented_df_pd.to_csv("./imputed_shifted_block.csv", index=False)

In [10]:
augmented_df_pd.to_parquet("./imputed_shifted_block.parquet", index=False)